In [ ]:
import pandas as pd
from src.config import *
import re
import numpy as np
from datetime import datetime
import streamlit as st


In [ ]:
data=pd.read_csv(DIR_WEIGHTED_DATA)
survey = pd.read_excel(DIR_META_DATA, sheet_name='survey')
choices = pd.read_excel(DIR_META_DATA, sheet_name='choices')

### Questions Table: Questions tab rows + columns setup

In [ ]:
TABLE_COLUMNS=[
    'Question',
    'All Answers',
    'USE',
    'AGE_GROUP',
    'GENDER',
    'URBAN_RURAL'
    ]

QUESTIONS=[
 # Stockouts

    # NONUSERS
    # health-seeking behaviour
    'nonusers_seek_contraceptives',
        # if yes
        'stockouts_nonusers',
        'stockouts_nonusers_response',
        'stockouts_nonusers_response_other',
        # if yes
    'health_facilities_visited',
    'health_facilities_other',

    # 'reason_no_visits', # No responses for this question

    # travel
    'transport_mode_nonusers',
    'travel_time_nonusers',
    
    # costs
    'nonuser_cost',
    
    # USERS
    'stockouts_users',
    'stockouts_response',
    'stockouts_response_other',

    # 
    'health_facilities_users', 
    'health_facilities_other_users',

    # travel
    'transport_mode_users', 
    'transport_mode_users_other',
    'travel_time_users',
    
    # costs
    'user_costs',

    # BOTH
    'willingness_to_travel'
    ]



### Questions Table: Radio/Settlements tab rows + columns setup

In [ ]:
LOCATIONS = [location for location in data.location.dropna().unique()]
TABLE_LOCATION_COLUMNS=['Question', *LOCATIONS]

### Data Manipulation

In [ ]:
def value_counts_multiple_answers_weighted(var: pd.Series, weights: pd.Series) -> pd.Series:
    var_name = var.name
    df = pd.DataFrame({'answer': var, 'weight': weights})
    df = df.dropna(subset=['answer'])          
    df['answer'] = df['answer'].astype(str).str.split('|')
    df = df.explode('answer', ignore_index=True) 
    out = df.groupby('answer')['weight'].sum().sort_values(ascending=False)
    out.rename({"answer":var_name})
    out.index.name=var_name
    return out

def split_dribar_weighted(df, split_var,question, condition_var=None, condition_exclude=None):
  # Filter df if condition is specified (exclude unwanted values)
  df_filtered = df.copy()
  if condition_var is not None and condition_var in df.columns and condition_exclude is not None:
    df_filtered = df_filtered[~df_filtered[condition_var].isin(condition_exclude)]
  
  if df_filtered[question].nunique()==0:
     return pd.DataFrame()
  
  if split_var!="No Split" and split_var!="":
    # Weighted count per answer x split group, built directly: groupby().apply() returns a wide frame
    # (no `question` column) whenever every group has the same set of answers, which broke pivot
    answered = df_filtered.dropna(subset=[question])
    long = pd.DataFrame({question: answered[question].astype(str).str.split('|'),
                         split_var: answered[split_var], 'weight': answered['normalized_weight']}).explode(question)
    if long[split_var].nunique() <= 1:
          var=str(long[split_var].dropna().iloc[0]) if long[split_var].notna().any() else ""
          st.warning(f"The selected split is not possible, all interviews are {split_var} ="+var)
          return pd.DataFrame()
    ret=long.groupby([question, split_var])['weight'].sum().unstack(split_var)
    sums=df_filtered.groupby(split_var)["normalized_weight"].sum()
    return round(ret.div(sums,axis=1)*100,2)
  
  ret=round(100*value_counts_multiple_answers_weighted(df_filtered[question],df_filtered["normalized_weight"])/df_filtered["normalized_weight"].sum(),2)
  return ret.reset_index().rename(columns={"weight":question,question:""}).set_index("")


In [ ]:

def get_split_answers(data,split_var,var, condition_var=None, condition_exclude=None):
    df=split_dribar_weighted(data,split_var,var, condition_var, condition_exclude)
    if df.empty:
        return ""
    order = split_dribar_weighted(data, "", var, condition_var, condition_exclude).index
    df = df.reindex(order)
    df=df.fillna(0)
    df=df.apply(lambda col: str(col.name)+": "+col.astype(str) + '%' )
    df.reset_index(inplace=True)

    df['Combined_Column'] = df.apply(lambda row: ' \n '.join(row.astype(str)), axis=1)
    merged_str="\n\n".join(df.Combined_Column)+"\n\n"
    return merged_str


### Populate: Questions Table

In [ ]:
def get_unsplit_answers(data,var, condition_var=None, condition_exclude=None):
    df=split_dribar_weighted(data,"",var, condition_var, condition_exclude)
    if df.empty:
        return ""
    df=df.fillna(0)
    df=df.apply(lambda col: col.astype(str) + '%' )
    df.reset_index(inplace=True)
    df['Combined_Column'] = df.apply(lambda row: ' \n '.join(row.astype(str)), axis=1)
    merged_str="\n\n".join(df.Combined_Column)+"\n\n"
    return merged_str


In [ ]:
%%capture
# Combine stockouts_nonusers_response_other into stockouts_nonusers_response. 00_clean_data drops questions
# no one answered, so the _other column only exists if someone chose "Other" (-22).
data['stockouts_nonusers_response_combined'] = data['stockouts_nonusers_response'].fillna('')
if 'stockouts_nonusers_response_other' in data.columns:
    data['stockouts_nonusers_response_combined'] += '|' + data['stockouts_nonusers_response_other'].fillna('')
data['stockouts_nonusers_response_combined'] = data['stockouts_nonusers_response_combined'].str.strip('|').replace('', np.nan)

table=pd.DataFrame(columns=TABLE_COLUMNS)

# Define conditional questions - exclude certain response values
# For nonusers_seek_contraceptives: exclude those who never tried (value 0) or don't know/prefer not to say
conditional_questions = {
    'stockouts_nonusers': ('nonusers_seek_contraceptives', [0]),
    'stockouts_nonusers_response_combined': ('stockouts_nonusers', [1]),
}

skipped_questions = []   # not in the data (no one answered, so dropped in 00) -- listed in the next cell
for variable in QUESTIONS:
    # Skip the _other field since we combined it
    if variable == 'stockouts_nonusers_response_other':
        continue
    
    # Use combined field for response
    var_to_use = 'stockouts_nonusers_response_combined' if variable == 'stockouts_nonusers_response' else variable
    if var_to_use not in data.columns:
        skipped_questions.append(var_to_use)
        continue
    
    condition_info = conditional_questions.get(var_to_use, None)
    condition_var = condition_info[0] if condition_info else None
    condition_exclude = condition_info[1] if condition_info else None
    
    all_answers=get_unsplit_answers(data, var_to_use, condition_var, condition_exclude)
    use_split=get_split_answers(data, "use", var_to_use, condition_var, condition_exclude)
    age_split=get_split_answers(data, "age_group", var_to_use, condition_var, condition_exclude)
    gender_split=get_split_answers(data, "gender", var_to_use, condition_var, condition_exclude)
    urban_rural_split=get_split_answers(data, "urban_rural", var_to_use, condition_var, condition_exclude)
    table.loc[len(table)] = [variable, all_answers, use_split, age_split, gender_split, urban_rural_split]


In [ ]:
print(f"{len(table)} questions in the table; skipped (not in the data): {skipped_questions or 'none'}")

In [ ]:
# 2026-08-27: was mapping survey.hint here (enumerator coding
# instructions, optional and mostly blank) instead of survey.label (the
# actual question text) -- left 'Question' blank for most rows and
# showed instruction text instead of the question for the rest. Now
# keeps both: Question = the real label, Hint = the enumerator note
# (kept as its own column, not dropped) where one exists.
name_to_label = dict(zip(survey.name, survey.label))
name_to_hint = dict(zip(survey.name, survey.hint))

table['Hint'] = table.Question.map(name_to_hint)
table.Question = table.Question.map(name_to_label)
table = table[['Question', 'Hint'] + [c for c in table.columns if c not in ('Question', 'Hint')]]


### Final Table Export:

In [ ]:
today_date = datetime.today()
date_string = today_date.strftime("%y_%m_%d")
out_path = Path(str(DIR_SUPPLY_QUESTIONS_TABLE).format(date=date_string))
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    table.to_excel(writer, sheet_name="Questions", index=False, freeze_panes=(1, 0))
print(f"Saved: {out_path}")